## Developing an Automatic Text Summarizer (Extractive + Abstractive Comparison)


Step 1: Importing all the required libraries

In [49]:
# importing all the required libraries
# Core
import re
import numpy as np
import pandas as pd

# NLP
import nltk
import spacy

# NLTK
from nltk.tokenize import sent_tokenize

# Sklearn for TF-IDF and cosine similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Transformers for summarization
from transformers import pipeline

# Evaluation metrics
from rouge_score import rouge_scorer

# Downloading the required NLTK resources
nltk.download("punkt")

# Loading the Spacy model
nlp = spacy.load("en_core_web_sm")

#remove warnings
import warnings
warnings.filterwarnings("ignore")

[nltk_data] Downloading package punkt to C:\Users\Saif
[nltk_data]     Ullah\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Step 2: upload the dataset

In [50]:
# load the dataset
df = pd.read_csv("bbc_news_dataset.csv")

In [51]:
#checking the dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2225 entries, 0 to 2224
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    2225 non-null   object
dtypes: object(1)
memory usage: 17.5+ KB


In [52]:
#checking the first few rows of the dataset
df.head()

,text
0,"Wal-Mart, the largest US retailer, has said it..."
1,Brute force budget cuts or spending caps would...
2,Trouble-hit Mitsubishi Motors is in talks with...
3,Typically it takes about three years from when...
4,"More than 2,000 business and political leaders..."


Step 3: preprocessing the data

In [53]:
# preprocessing the text data
def preprocess_text(text):
    """
    Clean preprocessing:
    - Lowercase
    - Keep numbers (important for summaries)
    - Remove noise
    - Sentence tokenization
    - Lemmatization
    - Remove stopwords & punctuation
    """
    
    # Lowercase
    text = text.lower()
    
    # Keep numbers + letters + dot
    text = re.sub(r'[^a-zA-Z0-9.,!? ]', '', text)
    
    # Sentence tokenization
    sentences = sent_tokenize(text)
    
    cleaned_sentences = []
    
    # Lemmatization + remove stopwords & punctuation
    for sent in sentences:
        doc = nlp(sent)
        
        words = [
            token.lemma_
            for token in doc
            if not token.is_stop
            and not token.is_punct
        ]
        
        cleaned_sentences.append(" ".join(words))
    
    cleaned_text = " ".join(cleaned_sentences)
    
    return cleaned_text, sentences

In [54]:
#apply the preprocessing to the dataset

# Store only cleaned text
df["cleaned_text"] = df["text"].apply(lambda x: preprocess_text(x)[0])

# Store sentences separately (optional)
df["sentences"] = df["text"].apply(lambda x: preprocess_text(x)[1])

In [55]:
# show the maximum columns and rows
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

In [56]:
# checking the data after preprocessing
df.head()

,text,cleaned_text,sentences
0,"Wal-Mart, the largest US retailer, has said it...",walmart large retailer say december sale expec...,"[walmart, the largest us retailer, has said it..."
1,Brute force budget cuts or spending caps would...,brute force budget cut spending cap illserve n...,[brute force budget cuts or spending caps woul...
2,Trouble-hit Mitsubishi Motors is in talks with...,troublehit mitsubishi motors talk french carma...,[troublehit mitsubishi motors is in talks with...
3,Typically it takes about three years from when...,typically take year decision take new model hi...,[typically it takes about three years from whe...
4,"More than 2,000 business and political leaders...","2,000 business political leader globe arrive s...","[more than 2,000 business and political leader..."


Step 4: Creating Extractive summary block

In [57]:
# Extractive summarization function with TF-IDF + cosine similarity + sentence ranking

def extractive_summary(text, num_sentences=3):
    """
    Extractive summarization using TF-IDF + cosine similarity + sentence ranking
    (Fully aligned with project requirement)
    """

    # Step 1: sentence tokenization
    sentences = sent_tokenize(text)

    # safety check
    if len(sentences) <= num_sentences:
        return text, list(range(len(sentences)))

    # Step 2: clean sentences (light cleaning only)
    cleaned_sentences = [
        re.sub(r'[^a-zA-Z0-9 ]', '', sent.lower())
        for sent in sentences
    ]

    # Step 3: TF-IDF matrix
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(cleaned_sentences)

    # Step 4: cosine similarity (allowed enhancement)
    similarity_matrix = cosine_similarity(tfidf_matrix)

    # Step 5: sentence ranking (core requirement)
    sentence_scores = similarity_matrix.sum(axis=1)

    ranked_sentences = np.argsort(sentence_scores)[::-1]

    selected_indices = sorted(ranked_sentences[:num_sentences])

    summary = " ".join([sentences[i] for i in selected_indices])

    return summary, selected_indices

In [58]:
# testing the extractive summarization function with a sample text from the dataset
sample_text = df["text"].iloc[0]
test_summary, test_indices = extractive_summary(sample_text)
print("✅ Extractive summary test passed!")
print(f"Summary: {test_summary[:200]}...")

✅ Extractive summary test passed!
Summary: Wal-Mart, the largest US retailer, has said its December sales are expected to be better than previously forecast because of strong post-Christmas sales."The continuing economic expansion, combined wi...


In [59]:
# if the test passed, apply the extractive summarization to the dataset
df["extractive_summary"] = df["text"].apply(lambda x: extractive_summary(x)[0])

In [60]:
# Take one sample to test the extractive summarization
text = df["text"].iloc[0]

Step 5: Loading the Facebook (bart-large-cnn) model from the transformer for Abstractive Summary

In [61]:
# load the t5 model for abstractive summarization using transformers pipeline

summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn"
)

'(ProtocolError('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)), '(Request ID: 01d56bee-b063-4213-8f83-73ee4a873652)')' thrown while requesting HEAD https://huggingface.co/facebook/bart-large-cnn/resolve/main/config.json
Retrying in 1s [Retry 1/5].
'(ProtocolError('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)), '(Request ID: 799f09f5-c010-43ba-b5f6-cea4f9ac85f7)')' thrown while requesting HEAD https://huggingface.co/facebook/bart-large-cnn/resolve/main/config.json
Retrying in 2s [Retry 2/5].
'(ProtocolError('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)), '(Request ID: 3440bde5-50ab-4eef-802d-73e5fe86f337)')' thrown while requesting HEAD https://huggingface.co/facebook/bart-large-cnn/resolve/main/config.json
Retrying in 4s [R

Step 6: Length control function for Abstractive summary

In [62]:
# this function will control the summary length based on the option provided by the user

def get_summary_length(option):

    """
    Controls summary size
    """

    option = option.lower()

    if option == "short":

        return 40, 15, 2

    elif option == "medium":

        return 80, 30, 3

    elif option == "long":

        return 120, 50, 5

    else:

        # Default
        return 80, 30, 3

Step 7: Creating Abstractive summary block

In [100]:
# this function will generate the abstractive summary using the BART model.
# It also handles large input text using chunk-based processing.

def abstractive_summary(text, max_len=130, min_len=40):

    """
    Generate abstractive summary using facebook/bart-large-cnn.

    Improvements:
    -------------------
    1. Handles large text inputs safely
    2. Uses chunk-based summarization
    3. Prevents token overflow issues
    4. Produces high-quality coherent summaries

    Parameters:
    -----------
    text : str
        Original input text

    max_len : int
        Maximum summary length

    min_len : int
        Minimum summary length

    Returns:
    --------
    final_summary : str
        Combined abstractive summary
    """

    # STEP 1: SPLIT TEXT INTO SENTENCES
    # =====================================================
    sentences = sent_tokenize(text)

    # STEP 2: BUILD CHUNKS (SAFE FOR BART)
    # =====================================================
    chunks = []
    current_chunk = ""

    # BART safe character limit per chunk
    chunk_size_limit = 400

    for sent in sentences:

        if len(current_chunk) + len(sent) < chunk_size_limit:
            current_chunk += " " + sent
        else:
            chunks.append(current_chunk.strip())
            current_chunk = sent

    # add remaining text
    if current_chunk:
        chunks.append(current_chunk.strip())

    # STEP 3: GENERATE SUMMARY FOR EACH CHUNK
    # =====================================================
    chunk_summaries = []

    for chunk in chunks:

        # BART does NOT need "summarize:" prefix (important difference)
        result = summarizer(
            chunk,
            max_length=max_len,
            min_length=min_len,
            max_new_tokens=80,
            do_sample=False,
            num_beams=3,
            repetition_penalty=2.0,
            no_repeat_ngram_size=3,
            length_penalty=1.2,
            early_stopping=True
        )

        chunk_summaries.append(result[0]["summary_text"])

    # STEP 4: MERGE CHUNK SUMMARIES
    # =====================================================
    combined_summary = " ".join(chunk_summaries)

    # STEP 5: FINAL REFINEMENT (IF NEEDED)
    # =====================================================
    if len(combined_summary) > 1200:

        final_result = summarizer(
            combined_summary[:1200],
            max_length=max_len,
            min_length=min_len,
            do_sample=False
        )

        final_summary = final_result[0]["summary_text"]

    else:
        final_summary = combined_summary

    return final_summary

Step 8: Highlight Selected Sentences

In [101]:
# highlight the important sentences in the original text based on the selected indices from the extractive summarization

def highlight_sentences(text, selected_indices):

    """
    Highlight important sentences in the original text based on extractive summarization.

    Purpose:
    - Visually distinguish sentences that were selected by the extractive model
    - Helps in comparing full text vs summary selection
    - Useful for analysis and debugging of TF-IDF ranking

    Parameters:
    - text (str): Original input text
    - selected_idx (list): Indices of sentences selected by extractive summarizer

    Returns:
    - str: Full text where selected sentences are marked as "[SELECTED]"
    """

    # Split original text into sentences
    sentences = sent_tokenize(text)

    highlighted_text = []

    # Iterate through sentences and highlight selected ones
    for index, sentence in enumerate(sentences):

        # Check if current sentence index is in the selected indices
        if index in selected_indices:

            highlighted_text.append(
                f"[IMPORTANT] {sentence}"
            )
        #otherwise, keep the sentence as is
        else:

            highlighted_text.append(sentence)

# Join all sentences back into a single string
    return "\n".join(highlighted_text)

Step 9: Adding ROUGE Score Evaluation Function

In [102]:
# rouge_score evaluation function, compute ROUGE scores between reference and generated summaries.
def compute_rouge(reference, generated):
    """
    Evaluate summary quality using ROUGE metrics.

    ROUGE measures similarity between:
    - reference summary (ground truth)
    - generated summary (model output)

    Metrics used:
    - ROUGE-1: unigram overlap
    - ROUGE-2: bigram overlap
    - ROUGE-L: longest common subsequence

    Returns:
    - dictionary of ROUGE scores
    """

    scorer = rouge_scorer.RougeScorer(
        ['rouge1', 'rouge2', 'rougeL'],
        use_stemmer=True
    )

    scores = scorer.score(reference, generated)

    return scores

Step 10: Save Summaries to file

In [103]:
# Save summaries to TXT and CSV files for easy access and analysis

def save_summaries(original,
                   extractive,
                   abstractive):

    # Save TXT file
    with open(
        "summary_output.txt",
        "w",
        encoding="utf-8"
    ) as file:

        file.write("ORIGINAL TEXT\n\n")
        file.write(original)

        file.write("\n\n")
        file.write("EXTRACTIVE SUMMARY\n\n")
        file.write(extractive)

        file.write("\n\n")
        file.write("ABSTRACTIVE SUMMARY\n\n")
        file.write(abstractive)

    # Save CSV file
    data = {

        "Original_Text": [original],
        "Extractive_Summary": [extractive],
        "Abstractive_Summary": [abstractive]
    }

    output_df = pd.DataFrame(data)

    output_df.to_csv(
        "summary_output.csv",
        index=False
    )

    print("Summaries saved successfully.")

Step 11: Main Testing Section

In [104]:
# Testing the main system with a sample text from the dataset

# Select sample text
text = df["text"].iloc[0]

# Choose summary length
length_option = "medium"

# Get length parameters
max_len, min_len, ext_sentences = get_summary_length(
    length_option
)

# Generate Extractive Summary
extractive_result, selected_indices = extractive_summary(
    text,
    num_sentences=ext_sentences
)

# Generate Abstractive Summary
abstractive_result = abstractive_summary(
    text,
    max_len=max_len,
    min_len=min_len
)

# Highlight sentences
highlighted_output = highlight_sentences(
    text,
    selected_indices
)

# Display Results
print("=" * 70)
print("ORIGINAL TEXT")
print("=" * 70)

print(text)

print("\n")

print("=" * 70)
print("EXTRACTIVE SUMMARY")
print("=" * 70)

print(extractive_result)

print("\n")

print("=" * 70)
print("ABSTRACTIVE SUMMARY")
print("=" * 70)

print(abstractive_result)

print("\n")

print("=" * 70)
print("HIGHLIGHTED IMPORTANT SENTENCES")
print("=" * 70)

print(highlighted_output)

Your max_length is set to 80, but your input_length is only 32. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=16)
Both `max_new_tokens` (=80) and `max_length`(=80) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=80) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ORIGINAL TEXT
Wal-Mart, the largest US retailer, has said its December sales are expected to be better than previously forecast because of strong post-Christmas sales."The continuing economic expansion, combined with job growth, has consumers ending this year on a high note," said Lynn Franco, director of the Conference Board's consumer research centre.The feel-good factor among US consumers rose in December for the first time since July according to new data.Consumers' confidence in the state of the US economy is at its highest for five months and they are optimistic about 2005, an influential survey says.US retailers have reported strong sales over the past 10 days after a slow start to the crucial festive season.


EXTRACTIVE SUMMARY
Wal-Mart, the largest US retailer, has said its December sales are expected to be better than previously forecast because of strong post-Christmas sales."The continuing economic expansion, combined with job growth, has consumers ending this year on a hi

Step 12: ROGUE Evaluation sample <br>
Note: Since our dataset does not contain human summaries, we use extractive summary as reference only for demonstration

In [105]:
#  rouge_score evaluation example
reference_summary = extractive_result

# Evaluate abstractive summary
rouge_scores = compute_rouge(
    reference_summary,
    abstractive_result
)

print("=" * 70)
print("ROUGE SCORES")
print("=" * 70)

print(rouge_scores)

ROUGE SCORES
{'rouge1': Score(precision=0.9375, recall=0.5, fmeasure=0.6521739130434783), 'rouge2': Score(precision=0.873015873015873, recall=0.46218487394957986, fmeasure=0.6043956043956044), 'rougeL': Score(precision=0.84375, recall=0.45, fmeasure=0.5869565217391305)}


Step 13: Save output file

In [106]:
# Save the summaries to an output file for future reference and analysis

save_summaries(
    text,
    extractive_result,
    abstractive_result
)

Summaries saved successfully.


Step 13: Custom User Input Features

In [107]:
# Custom user input for summarization

custom_text = input("Enter your text:\n")

if not custom_text.strip():
    print("No input provided!")
else:
    length_option = input("Choose summary length (short/medium/long): ")

    max_len, min_len, ext_sentences = get_summary_length(length_option)

    print("\nProcessing extractive summary...")
    ext_summary, selected_idx = extractive_summary(
        custom_text,
        num_sentences=ext_sentences
    )

    print("Processing abstractive summary...")
    abs_summary = abstractive_summary(
        custom_text,
        max_len=max_len,
        min_len=min_len
    )

    print("\n" + "=" * 70)
    print("EXTRACTIVE SUMMARY")
    print("=" * 70)
    print(ext_summary)

    print("\n" + "=" * 70)
    print("ABSTRACTIVE SUMMARY")
    print("=" * 70)
    print(abs_summary)

No input provided!


Step 14: Simple CLI Menu

In [108]:
#simple CLI menu for user to choose between dataset text and custom text for summarization

print("\n")
print("1. Use Dataset Text")
print("2. Use Custom Text")

choice = input("Enter your choice: ")

if choice == "1":

    text = df["text"].iloc[0]

elif choice == "2":

    text = input("Enter your custom text:\n")

else:

    print("Invalid choice")



1. Use Dataset Text
2. Use Custom Text
Invalid choice


In [109]:
# Ground-truth reference summary (if available in dataset)
# In many datasets, a human-written summary is provided for evaluation
# If not available, ROUGE evaluation cannot be performed

reference = None  


# =========================================================
# ROUGE EVALUATION SECTION
# =========================================================
# ROUGE is used to compare machine-generated summary with a reference (human summary)
# It measures overlap between words and phrases

if reference is not None:

    # Compute ROUGE scores for Extractive Summary
    rouge_ext = compute_rouge(reference, ext_summary)

    # Compute ROUGE scores for Abstractive Summary
    rouge_abs = compute_rouge(reference, abs_summary)

    # Display evaluation results
    print("\n========== ROUGE SCORES ==========\n")

    # Print Extractive summarization performance
    print("Extractive:", rouge_ext)
    print("Abstractive:", rouge_abs)